# Polynomial-charge field kernel (the order >= 2 field)

> **Executable notebook.** This is the runnable companion to the HDiv-VIM
> polynomial-charge field kernel. Every claim below (machine-precision closed
> forms, the zero-curvature flat limits, the uniform-sphere `-M/3`, convergence
> vs Gauss / Duffy references) is *recomputed live* in the code cells -- the
> embedded outputs are the real numbers, not assertions. The method itself lives
> in [`src/radia/vim/_field.py`](../../src/radia/vim/_field.py) (Python reference)
> and `src/core/rad_hdiv_vim.cpp` (the C++ fast path); the golden tests are
> [`validation_test/feec/test_hdiv_vim_poly_field.py`](../../validation_test/feec/test_hdiv_vim_poly_field.py).

**Status:** the whole Step 1 -> 2 -> 3 pipeline is in place (36 golden tests).
**Step 1** external field (`reconstruct_field_polynomial`); the **analytic
charge-field kernel** at degree 0/1/2 (closed form) AND arbitrary degree (general
assembler), flat-tet + affine-hex + curved-surface, with a **C++ port** of the
degree-1/2 kernels; **Step 2** the fast charge-coefficient assembly
`assemble_demag_field` (sums the C++ kernels, exact at any r); **Step 3** the
field-based nonlinear demag solve `solve_demag_picard` (under-relaxed Picard,
centroid collocation -> HDiv projection) -- validated on the uniform sphere
(linear chi -> the analytic `M = chi*H_ext/(1 + chi/3)`; mild saturating -> the
1-D scalar demag fixed point).
**Remaining:** the genuine order >= 2 NON-uniform projection (H_demag at quad
points -> VectorL2 order-p -> HDiv), a convex B-input (A-formulation) solve for
STIFF / strongly-saturating materials (the H-input Picard is stiff there --
effective chi >> 3 at the knee), curved/distorted-hex VOLUME, and an H-matrix
accelerated / C++ assembly for scale.

The kernel is **element-type agnostic**: the quadrature points / weights /
normals come from NGSolve's own `mesh.GetTrafo` + `IntegrationRule(el.type)` +
`specialcf.normal`, so the *same* code handles tet **and** hex (and prism)
meshes, flat or curved (`mip.measure` carries the curved Jacobian,
`specialcf.normal` the curved outward normal). A hex box and a tet box of the
same body give the same external field to ~machine precision
(`test_hex_matches_tet_*`).

## Why

A genuine order >= 2 nonlinear HDiv-VIM solve needs the magnetic field of a
**polynomial** magnetization `M(x)` (HDiv order *p*) -- both for the engineering
deliverable (the stray field around a soft-iron part) and for the constitutive
law `M = chi(H) H` inside the body.

The committed `reconstruct_field` uses the per-element **centroid** `M`
(piecewise-constant), i.e. the **surface charge `sigma = M.n` only** -- it
silently **drops the volume charge `rho = -div M`**. That is exact for uniform
`M` (`div M = 0`) but, wherever `div M != 0`, omitting `rho` is a **90-230%
error** (measured,
`tests/feec/test_hdiv_vim_poly_field.py::test_volume_charge_is_essential_for_div_M`).

This is also why the old order >= 2 nonlinear solve diverged:
`M_mass^-1 N m` (the weak demag field) has a solenoidal nullspace at order >= 2,
and even the centroid field reconstruction misses `rho`.

## The kernel

The field of a magnetization is the field of its magnetic charges
($H = -\nabla\phi_M$):

$$\rho = -\nabla\cdot M \quad(\text{volume charge, } L_2 \text{ order } p-1),
\qquad
\sigma = M\cdot n \quad(\text{surface charge, SurfaceL2 order } p)$$

$$H(r) = \frac{1}{4\pi}\left[\;
\int_V \rho\,\frac{r-r'}{|r-r'|^3}\,dV'
\;+\;
\int_S \sigma\,\frac{r-r'}{|r-r'|^3}\,dS'
\;\right]$$

These are exactly the charges the HDiv-VIM already forms in `ChargeGram`
(the `B` map: `rho = -div M` in `L2(p-1)`, `sigma = M.n` in `SurfaceL2(p)`). The
charge **Gram** `G` is the charge-charge **energy** $\iint q\,q'/|r-r'|$; this
kernel is its **field-at-a-point** companion $\int q\,(r-r')/|r-r'|^3$. The C++
already has the *constant*-charge potential building blocks (`_hdiv_phi_tet`,
`_hdiv_tri_potential` = Wilton); the field/polynomial generalisation is the
work.

## Staging

| Step | Scope | Singular? | Status |
|------|-------|-----------|--------|
| **1** | **External** points (stray field of a polynomial-M body), **tet + hex** | no (`r` clear of the body) | **done** -- `reconstruct_field_polynomial`, element-agnostic Python reference, golden-locked |
| **2** | Internal / near points (the field at the body's own quadrature points), **tet** | yes (`1/r^2` at `r'->r`) | **assembled** -- `reconstruct_field_internal` (self-volume spherical + far-volume + analytic surface), golden-locked; polynomial surface sigma / curved faces / C++ remain |
| 3 | Wire Step 2 into the per-element nonlinear Newton (`set_field` <= polynomial field, not `M_mass^-1 N m`) | -- | designed: genuine order >= 2 nonlinear M, golden vs a finer-mesh HDiv-VIM reference |

## Setup

Imports and the small **Gauss / Duffy reference integrators** used as oracles
throughout (these recompute each integral by brute-force quadrature, independent
of the analytic kernels they validate). `rel(a, b)` is the relative vector error
`||a - b|| / ||b||`. All prints are ASCII (Windows cp932-safe).

In [1]:
import numpy as np
from math import pi
import radia._radia_pybind as rp
from radia.vim import (
    reconstruct_field_polynomial, flat_triangle_charge_field,
    tet_self_volume_field, triangle_potential_const, triangle_potential_moment,
    tet_newtonian_potential, tet_volume_field_linear, linear_triangle_charge_field,
    tet_volume_field_quadratic, quadratic_triangle_charge_field,
    polynomial_triangle_charge_field, tet_volume_field_polynomial,
    hex_volume_field_linear, hex_volume_field_quadratic,
    make_t6_surface_map, curved_triangle_charge_field,
    make_t10_tet_map, curved_tet_volume_field, assemble_demag_field,
)

# canonical unit tet, a symmetric quadratic coefficient matrix Q/S
TET = np.array([[0, 0, 0], [1, 0, 0], [0, 1, 0], [0, 0, 1]], float)
QSYM = np.array([[0.3, 0.1, -0.2], [0.1, -0.4, 0.15], [-0.2, 0.15, 0.25]])


def rel(a, b):
    a = np.asarray(a, float); b = np.asarray(b, float)
    return np.linalg.norm(a - b) / np.linalg.norm(b)


def tri_field_gauss(P, r, nq=44):
    """Fine Duffy-Gauss reference for INT_T (r-r')/|r-r'|^3 dS' (uniform sigma)."""
    P = np.asarray(P, float); r = np.asarray(r, float)
    x, w = np.polynomial.legendre.leggauss(nq); s = 0.5 * (x + 1); ws = 0.5 * w
    e1, e2 = P[1] - P[0], P[2] - P[0]; area2 = np.linalg.norm(np.cross(e1, e2))
    F = np.zeros(3)
    for u, wu in zip(s, ws):
        for v, wv in zip(s, ws):
            q = P[0] + u * e1 + (v * (1 - u)) * e2; d = r - q
            F += (wu * wv * (1 - u) * area2) * d / np.linalg.norm(d) ** 3
    return F


def tri_charge_gauss(P, r, fn, nq=50):
    """Fine reference for INT_T sigma(r') (r-r')/|r-r'|^3 dS', sigma = fn(point)."""
    P = np.asarray(P, float); r = np.asarray(r, float)
    J = np.linalg.norm(np.cross(P[1] - P[0], P[2] - P[0]))
    xs, ws = np.polynomial.legendre.leggauss(nq); xs = 0.5 * (xs + 1); ws = 0.5 * ws
    F = np.zeros(3)
    for i in range(nq):
        for j in range(nq):
            u, v = xs[i], xs[j]; jac = (1 - u)
            q = P[0] + u * (P[1] - P[0]) + v * (1 - u) * (P[2] - P[0]); d = r - q
            F += ws[i] * ws[j] * jac * J * fn(q) * d / np.linalg.norm(d) ** 3
    return F


def tet_charge_gauss(V, r, fn, nq=24):
    """Fine reference for INT_V rho(r') (r-r')/|r-r'|^3 dV', rho = fn(point)."""
    V = np.asarray(V, float); r = np.asarray(r, float)
    xs, ws = np.polynomial.legendre.leggauss(nq); xs = 0.5 * (xs + 1); ws = 0.5 * ws
    V6 = abs(np.linalg.det(np.array([V[1] - V[0], V[2] - V[0], V[3] - V[0]])))
    F = np.zeros(3)
    for i in range(nq):
        for j in range(nq):
            for k in range(nq):
                a, b, c = xs[i], xs[j], xs[k]
                l1, l2, l3 = a, b * (1 - a), c * (1 - a) * (1 - b); jac = (1 - a) ** 2 * (1 - b)
                q = V[0] + l1 * (V[1] - V[0]) + l2 * (V[2] - V[0]) + l3 * (V[3] - V[0]); d = r - q
                F += ws[i] * ws[j] * ws[k] * jac * V6 * fn(q) * d / np.linalg.norm(d) ** 3
    return F


print("setup ready; radia._radia_pybind has the C++ probes:",
      all(hasattr(rp, n) for n in ["_hdiv_tri_field", "_hdiv_tet_field",
          "_hdiv_lin_tri_field", "_hdiv_quad_tri_field",
          "_hdiv_tet_volfield_linear", "_hdiv_tet_volfield_quadratic"]))

setup ready; radia._radia_pybind has the C++ probes: True


## Step 1 -- external field, validated (`reconstruct_field_polynomial`)

- **Uniform-M sphere** (`div M = 0`): center `H = -M/3` to ~1.2e-3; external =
  analytic dipole to ~6 % (= the flat-mesh faceting at `h = 0.4`, removed by
  `mesh.Curve` -- *not* a kernel error).
- **Linear M** (`M = (0, 0, M0(1 + z))`, `div M = M0`): the full kernel is
  coarse->fine self-convergent, while dropping `rho` (surface-only) is 90-230 %
  wrong -- the volume-charge term is essential.

The reference (Python) implementation samples the charges once over
(elements x Gauss-Duffy) and sums vectorised over the observation points; the
cost is independent of the number of observation points.

The cell below meshes a unit sphere with uniform `M = M0 e_z`, then a sphere with
**linear** `M` to show the volume charge is essential.

In [2]:
import ngsolve as ng
from netgen.csg import CSGeometry, Sphere, Pnt


def sphere_mesh(h):
    g = CSGeometry(); g.Add(Sphere(Pnt(0, 0, 0), 1.0))
    with ng.TaskManager():
        return ng.Mesh(g.GenerateMesh(maxh=h))


def H_dipole(r, m):
    rn = np.linalg.norm(r); rh = r / rn; mv = np.array([0.0, 0.0, m])
    return (1.0 / (4 * pi)) * (3.0 * np.dot(mv, rh) * rh - mv) / rn ** 3


# --- uniform M sphere: center -M/3 and external dipole ---
Mval = 5.0e5
mesh = sphere_mesh(0.4)
fes = ng.HDiv(mesh, order=1); gf = ng.GridFunction(fes)
with ng.TaskManager():
    gf.Set(ng.CoefficientFunction((0, 0, Mval)))
    Hc = reconstruct_field_polynomial(mesh, gf, np.array([[0.0, 0.0, 0.0]]), quad=4)
    obs = np.array([[0, 0, 2.0], [0, 0, 3.0], [2.0, 0, 0.0]], float)
    Hext = reconstruct_field_polynomial(mesh, gf, obs, quad=4)

print("uniform-M sphere, ne =", mesh.GetNE(ng.VOL))
print(f"  center H_z = {Hc[0,2]:.5e}   -M/3 = {-Mval/3:.5e}   rel = {abs(Hc[0,2]+Mval/3)/(Mval/3):.2e}")
m_dip = Mval * (4 * pi / 3)
for i, r in enumerate(obs):
    print(f"  external r={r.tolist()}  vs dipole  rel = {rel(Hext[i], H_dipole(r, m_dip)):.2e}  (flat-mesh faceting ~6%)")

uniform-M sphere, ne = 260
  center H_z = -1.66461e+05   -M/3 = -1.66667e+05   rel = 1.23e-03
  external r=[0.0, 0.0, 2.0]  vs dipole  rel = 5.53e-02  (flat-mesh faceting ~6%)
  external r=[0.0, 0.0, 3.0]  vs dipole  rel = 5.59e-02  (flat-mesh faceting ~6%)
  external r=[2.0, 0.0, 0.0]  vs dipole  rel = 5.87e-02  (flat-mesh faceting ~6%)


In [3]:
# --- linear M (div M = M0 != 0): the volume charge is ESSENTIAL ---
M0 = 3.0e5
obs2 = np.array([[0, 0, 2.5], [2.5, 0, 0.0], [1.5, 1.5, 1.0]], float)


def linM_field(h, drop_rho):
    me = sphere_mesh(h); fe = ng.HDiv(me, order=1); g = ng.GridFunction(fe)
    with ng.TaskManager():
        g.Set(ng.CoefficientFunction((0, 0, M0 * (1 + ng.z))))
        return reconstruct_field_polynomial(me, g, obs2, quad=4, include_volume=not drop_rho)


Hfull_c = linM_field(0.5, drop_rho=False)   # coarse, full kernel
Hdrop_c = linM_field(0.5, drop_rho=True)    # coarse, surface-only (drops rho)
Hfull_f = linM_field(0.3, drop_rho=False)   # fine, full kernel (reference)
print("linear-M sphere (div M = M0): full kernel coarse->fine vs surface-only (drop rho)")
for i in range(len(obs2)):
    rel_cf = rel(Hfull_c[i], Hfull_f[i])     # full kernel self-convergence
    rel_drop = rel(Hdrop_c[i], Hfull_f[i])   # error from dropping the volume charge
    print(f"  obs {i}: full coarse->fine = {rel_cf:.2e}   drop-rho error = {rel_drop*100:.0f}%")

linear-M sphere (div M = M0): full kernel coarse->fine vs surface-only (drop rho)
  obs 0: full coarse->fine = 5.35e-02   drop-rho error = 91%
  obs 1: full coarse->fine = 4.94e-02   drop-rho error = 233%
  obs 2: full coarse->fine = 5.01e-02   drop-rho error = 166%


## Step 2 -- internal / near singular field

For a query point `r` inside element `e`, split the charge sum by proximity:

- **far** (elements/faces not containing or adjacent to `r`): non-singular -> the
  Step-1 quadrature.
- **self / near** (the element holding `r`, and `r`'s own faces): singular ->
  handled analytically.

**Kernel A -- self-element VOLUME charge, spherical ray-trace**
(`tet_self_volume_field`). Substituting $r' = r + s\,\hat s$
($dV' = s^2\,ds\,d\Omega$, $r-r' = -s\,\hat s$, $|r-r'|^3 = s^3$) cancels the
kernel:

$$H_{\text{self}}(r) = \frac{1}{4\pi}\int_e \rho\,\frac{r-r'}{|r-r'|^3}\,dV'
= -\frac{1}{4\pi}\int_{S^2}\hat s
\left[\int_0^{s_{\max}(\hat s)} \rho(r + s\hat s)\,ds\right]d\Omega
\qquad(\text{NON-singular})$$

`smax(s_hat)` = ray distance from `r` to the element boundary; the inner
$\int\rho\,ds$ is closed-form for a polynomial `rho`. Golden: constant `rho` on
the unit tet vs $-(\rho/4\pi)\nabla\Phi_{\text{tet}}$ (FD of the exact analytic
Newtonian potential `_hdiv_phi_tet`) at interior points -> rel 7e-4 ... 3e-3.

**Kernel B -- surface charge, analytic uniform-triangle field**
(`flat_triangle_charge_field`). The exact $\int_T (r-r')/|r-r'|^3\,dS'$ for a
flat triangle (Wilton/Graglia: solid-angle normal term + per-edge log tangential
term), valid at any `r` (near/far/on-face PV).

The cell below validates **Kernel B** vs a fine Gauss reference (machine
precision), and **Kernel A** vs the closed-form `tet_volume_field_linear` it
converges to (the spherical method's own ~1e-3 accuracy).

In [4]:
# Kernel B: analytic uniform-triangle field == fine Gauss, to machine precision
P = TET[[0, 1, 2]]
print("Kernel B -- flat_triangle_charge_field vs fine Gauss (uniform sigma):")
for r in [np.array([0.3, 0.3, 0.5]), np.array([0.3, 0.3, -0.5]), np.array([1.0, 1.0, 0.3])]:
    print(f"  r={r.tolist()}  rel = {rel(flat_triangle_charge_field(P, r), tri_field_gauss(P, r)):.2e}")

# Kernel A: spherical ray-trace (self volume charge) == the closed form it converges to
print("Kernel A -- tet_self_volume_field (spherical) vs the closed form (interior r):")
gg = np.array([0.7, -0.3, 0.5]); rho0 = 0.4
r = np.array([0.25, 0.25, 0.25])
Fclosed = tet_volume_field_linear(TET, r, rho0, gg)
Fsph = 4 * np.pi * tet_self_volume_field(TET, r, lambda p: rho0 + float(np.dot(gg, p)),
                                         nth=48, nph=96, ns=10)
print(f"  r={r.tolist()}  rel = {rel(Fsph, Fclosed):.2e}  (spherical method's own ~1e-3 accuracy)")

Kernel B -- flat_triangle_charge_field vs fine Gauss (uniform sigma):
  r=[0.3, 0.3, 0.5]  rel = 6.34e-16
  r=[0.3, 0.3, -0.5]  rel = 6.34e-16
  r=[1.0, 1.0, 0.3]  rel = 2.36e-15
Kernel A -- tet_self_volume_field (spherical) vs the closed form (interior r):


  r=[0.25, 0.25, 0.25]  rel = 2.52e-03  (spherical method's own ~1e-3 accuracy)


## The polynomial volume-charge field -- degree 1 (closed form)

The order-2 volume charge `rho = -div M` is **linear** per cell, and its field is
a **closed form**, exact to machine precision at any point. The key identity is
$(r-r')/R^3 = \nabla'(1/R)$, so by the product rule + divergence theorem the
field of a polynomial volume charge reduces to **lower-degree potential
integrals** (one differential order down):

$$\int_V \rho\,\frac{r-r'}{R^3}\,dV'
= \sum_{\text{faces}} n_f \int_{\text{face}} \frac{\rho}{R}\,dS'
- \int_V \frac{\nabla\rho}{R}\,dV'$$

For linear $\rho = \rho_0 + g\cdot r'$ ($\nabla\rho = g$ const) this is
$\sum_f n_f\,[\rho_0 I_0^f + g\cdot M_1^f] - g\,\Phi_{\text{tet}}$, needing only:

- `triangle_potential_const` -- $I_0 = \int_T 1/R\,dS'$ (Wilton; bit-identical to
  the C++ `_hdiv_tri_potential`);
- `triangle_potential_moment` -- $M_1 = \int_T r'/R\,dS'$, first moment via the
  surface divergence theorem;
- `tet_newtonian_potential` -- $\Phi_{\text{tet}} = \int_V 1/R\,dV'
  = -\tfrac12\sum_f h_f I_0^f$ (matches the C++ `_hdiv_phi_tet` to machine
  precision).

`tet_volume_field_linear(verts, r, rho0, grho)` assembles these. Below: the
**constant** case vs the independently-derived $-\nabla\Phi_{\text{tet}}$ (central
FD), and the **linear** case vs far-tet Gauss.

In [5]:
# constant rho: closed form == -grad(PhiTet) (two independent derivations)
print("tet_volume_field_linear (constant rho) vs -grad(PhiTet) [central FD]:")
d = 1e-5
for r in [np.array([0.25, 0.25, 0.25]), np.array([0.4, 0.3, 0.1]), np.array([2.0, 0.0, 0.0])]:
    Fc = tet_volume_field_linear(TET, r, 1.0, np.zeros(3))
    g = np.zeros(3)
    for k in range(3):
        rp_, rm_ = r.copy(), r.copy(); rp_[k] += d; rm_[k] -= d
        g[k] = (tet_newtonian_potential(TET, rp_) - tet_newtonian_potential(TET, rm_)) / (2 * d)
    print(f"  r={r.tolist()}  rel = {rel(Fc, -g):.2e}")

# linear rho: closed form == far-tet Gauss to machine precision
print("tet_volume_field_linear (linear rho) vs far-tet Gauss:")
for r in [np.array([2.0, 0, 0]), np.array([1.5, 1.5, 1.0]), np.array([-1.0, 0.5, 0.5])]:
    Fc = tet_volume_field_linear(TET, r, rho0, gg)
    Fg = tet_charge_gauss(TET, r, lambda p: rho0 + np.dot(gg, p), 24)
    print(f"  r={r.tolist()}  rel = {rel(Fc, Fg):.2e}")

tet_volume_field_linear (constant rho) vs -grad(PhiTet) [central FD]:
  r=[0.25, 0.25, 0.25]  rel = 1.35e-09
  r=[0.4, 0.3, 0.1]  rel = 3.90e-10
  r=[2.0, 0.0, 0.0]  rel = 2.80e-10
tet_volume_field_linear (linear rho) vs far-tet Gauss:


  r=[2.0, 0.0, 0.0]  rel = 3.31e-14

  r=[1.5, 1.5, 1.0]  rel = 1.72e-14


  r=[-1.0, 0.5, 0.5]  rel = 8.16e-15


## The polynomial surface-charge field -- degree 1 (closed form)

The order-2 surface charge `sigma = M.n` is **linear** per face. Since
$(r-r')/R^3 = -\nabla_r(1/R)$, the field of a linear $\sigma$ is exactly
$-\nabla_r\phi_\sigma$ with $\phi_\sigma = \int_T \sigma/R\,dS' = \sigma_0 I_0 +
s\cdot M_1$ -- the degree-1 triangle potential. Differentiating in closed form:

$$\int_T (\sigma_0 + s\cdot r')\,\frac{r-r'}{R^3}\,dS'
= (\sigma_0 + s\cdot r_p)\,F_{\text{const}}
- \sum_{\text{edges}} (s\cdot m_e)\,G_e - I_0\,s_\parallel$$

needing only $F_{\text{const}}$ = `flat_triangle_charge_field`, $I_0$ =
`triangle_potential_const`, $s_\parallel$ = in-plane part of $s$, and one new
elementary building block $G_e = \int_{\text{edge}} (r-r')/R\,dl$
(closed-form `asinh`/`sqrt`). `linear_triangle_charge_field` assembles these;
`s = 0` reproduces $\sigma_0\,$`flat_triangle_charge_field` bit-identically. This
is the surface companion of `tet_volume_field_linear`.

In [6]:
s = np.array([0.6, -0.4, 0.3]); sig0 = 0.5
print("linear_triangle_charge_field (linear sigma) vs off-plane Gauss:")
for r in [np.array([0.3, 0.3, 0.7]), np.array([1.5, 0.5, 0.4]), np.array([-0.5, 0.4, 0.6])]:
    Fc = linear_triangle_charge_field(P, r, sig0, s)
    Fg = tri_charge_gauss(P, r, lambda p: sig0 + np.dot(s, p), 44)
    print(f"  r={r.tolist()}  rel = {rel(Fc, Fg):.2e}")

# s = 0 reproduces sigma0 * flat_triangle_charge_field bit-identically
r = np.array([0.3, 0.3, 0.7])
abs_diff = np.linalg.norm(linear_triangle_charge_field(P, r, sig0, np.zeros(3))
                          - sig0 * flat_triangle_charge_field(P, r))
print(f"s=0 reduction: ||linear(s=0) - sigma0*flat|| = {abs_diff:.2e}  (bit-identical)")

linear_triangle_charge_field (linear sigma) vs off-plane Gauss:
  r=[0.3, 0.3, 0.7]  rel = 2.60e-15
  r=[1.5, 0.5, 0.4]  rel = 2.68e-15
  r=[-0.5, 0.4, 0.6]  rel = 2.77e-15
s=0 reduction: ||linear(s=0) - sigma0*flat|| = 0.00e+00  (bit-identical)


## The polynomial volume-charge field -- degree 2 (closed form)

The quadratic volume charge $\rho = \rho_0 + g\cdot r' + r'^{\mathsf T} Q r'$
($Q$ symmetric) field, via the same divergence-theorem recursion:

$$\int_V \rho\,\frac{r-r'}{R^3}\,dV'
= \sum_{\text{faces}} n_f\,[\rho_0 I_0^f + g\cdot M_1^f + Q{:}M_2^f]
- (g\,\Phi_{\text{tet}} + 2\,Q\cdot V_1)$$

adds two degree-2 moment building blocks, each from the **same identities one
degree up**: `triangle_potential_moment2` ($M_2 = \int_T r'\otimes r'/R\,dS'$,
from the Hessian identity) and `tet_newtonian_moment`
($V_1 = \int_V r'/R\,dV'$). `tet_volume_field_quadratic` assembles these; `Q = 0`
reduces to `tet_volume_field_linear` bit-identically.

In [7]:
print("tet_volume_field_quadratic vs far-tet Gauss:")
for r in [np.array([2.0, 0, 0]), np.array([1.5, 1.5, 1.0]), np.array([-1.0, 0.5, 0.5])]:
    Fc = tet_volume_field_quadratic(TET, r, rho0, gg, QSYM)
    Fg = tet_charge_gauss(TET, r, lambda p: rho0 + np.dot(gg, p) + p @ QSYM @ p, 26)
    print(f"  r={r.tolist()}  rel = {rel(Fc, Fg):.2e}")

# Q = 0 reduces to tet_volume_field_linear bit-identically
r = np.array([0.25, 0.25, 0.25])
ad = np.linalg.norm(tet_volume_field_quadratic(TET, r, rho0, gg, np.zeros((3, 3)))
                    - tet_volume_field_linear(TET, r, rho0, gg))
print(f"Q=0 reduction: ||quadratic(Q=0) - linear|| = {ad:.2e}  (bit-identical)")

tet_volume_field_quadratic vs far-tet Gauss:


  r=[2.0, 0.0, 0.0]  rel = 4.13e-14


  r=[1.5, 1.5, 1.0]  rel = 2.17e-14


  r=[-1.0, 0.5, 0.5]  rel = 1.19e-14
Q=0 reduction: ||quadratic(Q=0) - linear|| = 0.00e+00  (bit-identical)


## The polynomial surface-charge field -- degree 2 (closed form)

The quadratic surface charge $\sigma = \sigma_0 + s\cdot r' + r'^{\mathsf T} S r'$
($S$ symmetric) field, via the systematic in-plane/normal split
$(r-r')/R^3 = \nabla'_s(1/R) + h\,n/R^3$:

$$\int_T \sigma\,\frac{r-r'}{R^3}\,dS'
= \underbrace{\sum_e m_e \int_{\text{edge}} \frac{\sigma}{R}\,dl
- (P s\,I_0 + 2\,P S\,M_1)}_{\text{in-plane}}
+ \underbrace{h\,n\,[\sigma_0 J^3_0 + s\cdot J^3_1 + S{:}J^3_2]}_{\text{normal}}$$

with the $1/R^3$ moments $J^3_0 = \int_T 1/R^3 = (n\cdot F_{\text{const}})/h$,
$J^3_1$, $J^3_2$, plus the quadratic edge integrals.
`quadratic_triangle_charge_field` assembles these; `S = 0` matches
`linear_triangle_charge_field` (two independent derivations -- in-plane/normal vs
$-\nabla\phi$ -- agreeing). It subsumes the constant and linear cases.

In [8]:
print("quadratic_triangle_charge_field vs off-plane Gauss:")
for r in [np.array([0.3, 0.3, 0.7]), np.array([1.5, 0.5, 0.4]), np.array([-0.5, 0.4, 0.6])]:
    Fc = quadratic_triangle_charge_field(P, r, sig0, s, QSYM)
    Fg = tri_charge_gauss(P, r, lambda p: sig0 + np.dot(s, p) + p @ QSYM @ p, 46)
    print(f"  r={r.tolist()}  rel = {rel(Fc, Fg):.2e}")

# S = 0 matches linear_triangle_charge_field (two independent derivations)
r = np.array([0.3, 0.3, 0.7])
print(f"S=0 reduction: quadratic(S=0) vs linear  rel = "
      f"{rel(quadratic_triangle_charge_field(P, r, sig0, s, np.zeros((3,3))), linear_triangle_charge_field(P, r, sig0, s)):.2e}")

quadratic_triangle_charge_field vs off-plane Gauss:
  r=[0.3, 0.3, 0.7]  rel = 1.50e-15
  r=[1.5, 0.5, 0.4]  rel = 3.91e-15
  r=[-0.5, 0.4, 0.6]  rel = 3.72e-15
S=0 reduction: quadratic(S=0) vs linear  rel = 1.33e-15


## Arbitrary degree (the general assembler)

The degree-0/1/2 closed forms are fast, hand-derived special cases. The
**general assembler** (`polynomial_triangle_charge_field`,
`tet_volume_field_polynomial`) handles **any polynomial degree** via the general
moment recursion, and reduces to the closed forms at degree <= 2 (two code paths
agree to machine precision).

- **Surface moments** $A_k = \int_T \xi^{\otimes k}/R$, $B_k = \int_T
  \xi^{\otimes k}/R^3$ from the master recursion; $A_k$ comes from $A_{k-2}$ +
  edge integrals alone (h-safe). `triangle_inplane_moments(P, r, degree)`.
- **Volume potential moments** $\int_V r'^\alpha/R$ from $1/R = \tfrac12
  \nabla'^2 R$ + Euler, bottoming at $\Phi_{\text{tet}}$, reducing to surface
  potentials; the **volume field** is the divergence-theorem recursion.

Below: cubic surface + cubic volume vs Gauss (machine precision), and the
degree-2 general path vs the closed form.

In [9]:
# cubic surface charge vs Gauss
rng = np.random.RandomState(3)
cf = {(ax, ay, k - ax - ay): rng.randn()
      for k in range(4) for ax in range(k + 1) for ay in range(k + 1 - ax)}
sigc = lambda p: sum(c * p[0] ** ax * p[1] ** ay * p[2] ** az for (ax, ay, az), c in cf.items())
print("polynomial_triangle_charge_field (cubic) vs off-plane Gauss:")
for r in [np.array([0.3, 0.3, 0.7]), np.array([1.5, 0.5, 0.4])]:
    print(f"  r={r.tolist()}  rel = {rel(polynomial_triangle_charge_field(P, r, sigc, 3), tri_charge_gauss(P, r, sigc, 52)):.2e}")

# general assembler at degree 2 == the closed form (two paths)
sig2 = lambda p: sig0 + np.dot(s, p) + p @ QSYM @ p
r = np.array([0.2, 0.2, 0.3])
print(f"degree-2 general vs closed-form  rel = {rel(polynomial_triangle_charge_field(P, r, sig2, 2), quadratic_triangle_charge_field(P, r, sig0, s, QSYM)):.2e}")

# cubic volume charge vs Gauss
rng2 = np.random.RandomState(4)
cf3 = {(ax, ay, k - ax - ay): rng2.randn()
       for k in range(4) for ax in range(k + 1) for ay in range(k + 1 - ax)}
rho3 = lambda p: sum(c * p[0] ** ax * p[1] ** ay * p[2] ** az for (ax, ay, az), c in cf3.items())
print("tet_volume_field_polynomial (cubic) vs far-tet Gauss:")
for r in [np.array([2.0, 0, 0]), np.array([-1.0, 0.5, 0.5])]:
    print(f"  r={r.tolist()}  rel = {rel(tet_volume_field_polynomial(TET, r, rho3, 3), tet_charge_gauss(TET, r, rho3, 24)):.2e}")

polynomial_triangle_charge_field (cubic) vs off-plane Gauss:
  r=[0.3, 0.3, 0.7]  rel = 2.47e-15


  r=[1.5, 0.5, 0.4]  rel = 1.28e-14


degree-2 general vs closed-form  rel = 4.29e-16
tet_volume_field_polynomial (cubic) vs far-tet Gauss:


  r=[2.0, 0.0, 0.0]  rel = 2.42e-12


  r=[-1.0, 0.5, 0.5]  rel = 1.51e-13


## C++ port of the degree-1/2 kernels (the order <= 2 fast path)

The degree-1/2 closed forms are ported to C++ (`src/core/rad_hdiv_vim.cpp`,
probes in `radia_pybind.cpp`): `TriMoment1`/`TriMoment2`, `TetMoment1`,
`TetVolFieldLinear`/`TetVolFieldQuadratic`, `LinTriField`/`QuadTriField` -- built
on the existing `TriPotential`/`TriField`/`PhiTet`/`TetField`. Each is validated
**entry-by-entry vs its Python reference to machine precision** (the lab pattern:
a C++ probe validated against the Python formula). This is the fast per-element
kernel for the practical order <= 2 case (`rho = -div M` linear,
`sigma = M.n` quadratic).

In [10]:
Vt = P.ravel().tolist(); Vv = TET.ravel().tolist()
Qf = QSYM.ravel().tolist(); gl = gg.tolist(); sl = s.tolist()
print("C++ degree-1/2 kernels vs Python reference (machine precision):")
for r in [np.array([0.3, 0.3, 0.7]), np.array([1.5, 0.5, 0.4]), np.array([0.2, 0.2, 0.05])]:
    rl = r.tolist()
    e_lin = rel(rp._hdiv_lin_tri_field(Vt, rl, sig0, sl), linear_triangle_charge_field(P, r, sig0, s))
    e_quad = rel(rp._hdiv_quad_tri_field(Vt, rl, sig0, sl, Qf), quadratic_triangle_charge_field(P, r, sig0, s, QSYM))
    print(f"  r={rl}  LinTriField rel={e_lin:.2e}   QuadTriField rel={e_quad:.2e}")
for r in [np.array([0.25, 0.25, 0.25]), np.array([2.0, 0, 0]), np.array([0.4, 0.3, 0.1])]:
    rl = r.tolist()
    e_vl = rel(rp._hdiv_tet_volfield_linear(Vv, rl, rho0, gl), tet_volume_field_linear(TET, r, rho0, gg))
    e_vq = rel(rp._hdiv_tet_volfield_quadratic(Vv, rl, rho0, gl, Qf), tet_volume_field_quadratic(TET, r, rho0, gg, QSYM))
    print(f"  r={rl}  TetVolFieldLinear rel={e_vl:.2e}   TetVolFieldQuadratic rel={e_vq:.2e}")

C++ degree-1/2 kernels vs Python reference (machine precision):
  r=[0.3, 0.3, 0.7]  LinTriField rel=3.81e-16   QuadTriField rel=1.07e-15
  r=[1.5, 0.5, 0.4]  LinTriField rel=5.41e-16   QuadTriField rel=1.04e-15
  r=[0.2, 0.2, 0.05]  LinTriField rel=9.02e-17   QuadTriField rel=2.54e-16
  r=[0.25, 0.25, 0.25]  TetVolFieldLinear rel=1.43e-15   TetVolFieldQuadratic rel=6.21e-16
  r=[2.0, 0.0, 0.0]  TetVolFieldLinear rel=3.87e-14   TetVolFieldQuadratic rel=6.99e-14
  r=[0.4, 0.3, 0.1]  TetVolFieldLinear rel=8.10e-16   TetVolFieldQuadratic rel=1.07e-15


## Flat-faced hex / prism (affine hex volume field)

The analytic volume field is polytope-general:
`polytope_volume_field_quadratic(boundary_tris, ...)` takes any list of
`(triangle, outward normal)`, so a hex's 6 **planar** quad faces (triangulated
into 12 triangles by `hex_boundary_triangles`, NGSolve vertex order) drive the
**same** face-loop as a tet's 4. `hex_volume_field_linear` /
`hex_volume_field_quadratic` are the hex analogues. Validated to **machine
precision** against the sum of analytic tet fields over a 6-tet (Kuhn)
decomposition of the same box (two independent analytic computations agree), and
a sheared affine parallelepiped matches a box-Gauss reference. EXACT for
**planar-faced** hexes; a trilinear (distorted) hex has **bilinear (non-planar)
faces** -> that is the curved case. (The hex *surface* charge field needs no new
code -- a boundary quad is two triangles.)

In [11]:
# NGSolve hex vertex order; 6-tet (Kuhn) decomposition sharing the v0-v6 diagonal
HEX = np.array([[0,0,0],[0,0,1],[0,1,1],[0,1,0],[1,0,0],[1,0,1],[1,1,1],[1,1,0]], float)
paths = [[0,4,5,6],[0,4,7,6],[0,3,7,6],[0,3,2,6],[0,1,2,6],[0,1,5,6]]
print("hex_volume_field_quadratic vs SUM of analytic tet fields (Kuhn decomposition):")
for r in [np.array([2.0, 0.5, 0.5]), np.array([0.5, 0.5, 2.0]), np.array([-1.0, 0.5, 0.5])]:
    hexQ = hex_volume_field_quadratic(HEX, r, rho0, gg, QSYM)
    tetQ = sum(tet_volume_field_quadratic(HEX[p], r, rho0, gg, QSYM) for p in paths)
    print(f"  r={r.tolist()}  rel = {rel(hexQ, tetQ):.2e}")

# sheared affine parallelepiped (planar faces) vs a box-Gauss reference
A = np.array([[1.0, 0.3, 0.1], [0.0, 1.2, 0.2], [0.0, 0.0, 0.9]])
Vp = (HEX @ A.T) + np.array([0.2, -0.1, 0.3])


def box_gauss(V8, r, rho_fn, nq=24):
    xs, ws = np.polynomial.legendre.leggauss(nq); xs = 0.5 * (xs + 1); ws = 0.5 * ws
    def hexmap(xi, eta, ze):
        x0, x1, y0, y1, z0, z1 = 1-xi, xi, 1-eta, eta, 1-ze, ze
        N = [x0*y0*z0, x0*y0*z1, x0*y1*z1, x0*y1*z0, x1*y0*z0, x1*y0*z1, x1*y1*z1, x1*y1*z0]
        return sum(N[i] * V8[i] for i in range(8))
    def detJ(xi, eta, ze):
        e = 1e-6
        px = (hexmap(xi+e, eta, ze) - hexmap(xi-e, eta, ze)) / (2*e)
        py = (hexmap(xi, eta+e, ze) - hexmap(xi, eta-e, ze)) / (2*e)
        pz = (hexmap(xi, eta, ze+e) - hexmap(xi, eta, ze-e)) / (2*e)
        return abs(np.dot(px, np.cross(py, pz)))
    F = np.zeros(3)
    for i in range(nq):
        for j in range(nq):
            for k in range(nq):
                q = hexmap(xs[i], xs[j], xs[k]); d = r - q
                F += ws[i]*ws[j]*ws[k]*detJ(xs[i], xs[j], xs[k])*rho_fn(q)*d/np.linalg.norm(d)**3
    return F


rhoQ = lambda p: rho0 + np.dot(gg, p) + p @ QSYM @ p
print("sheared affine hex vs box-Gauss (the ~1e-8 floor is the FD Jacobian in the reference):")
for r in [np.array([3.0, 0.5, 0.5]), np.array([0.5, 3.0, 0.5])]:
    print(f"  r={r.tolist()}  rel = {rel(hex_volume_field_quadratic(Vp, r, rho0, gg, QSYM), box_gauss(Vp, r, rhoQ, 26)):.2e}")

hex_volume_field_quadratic vs SUM of analytic tet fields (Kuhn decomposition):
  r=[2.0, 0.5, 0.5]  rel = 1.09e-14
  r=[0.5, 0.5, 2.0]  rel = 9.71e-15


  r=[-1.0, 0.5, 0.5]  rel = 1.01e-15
sheared affine hex vs box-Gauss (the ~1e-8 floor is the FD Jacobian in the reference):


  r=[3.0, 0.5, 0.5]  rel = 3.66e-11


  r=[0.5, 3.0, 0.5]  rel = 3.51e-11


## Curved triangular faces (surface, singularity subtraction)

A curved face has **no closed form**, so
`curved_triangle_charge_field(surf_map, r, sigma_fn, nq)` uses **singularity
subtraction**: at the surface projection $(u_0, v_0)$ of `r`, the flat **tangent
triangle** carries the **exact $1/r^2$ singularity** -> its field is the analytic
`flat_triangle_charge_field`; the smooth **(curved - tangent)** remainder is
integrated by a **Duffy-refined** rule (3 sub-triangles fanned from $(u_0, v_0)$,
each collapse-mapped so the Jacobian $\sim s$ kills the residual $1/\rho$).
`surf_map(u,v) -> (x, x_u, x_v)` (build from NGSolve `GetTrafo`;
`make_t6_surface_map` builds it for a 6-node quadratic patch).

This is **quadrature-refined**, not closed-form: rel err ~1e-6 at nq=20,
controllable by `nq`. The **zero-curvature limit reproduces
`flat_triangle_charge_field`**. *Lesson:* a naive brute-force Gauss reference does
**not** converge near a curved surface -- the Duffy fan is needed for both the
kernel and its reference.

In [12]:
from radia.vim._field import _project_to_surface

# zero-curvature T6 patch reproduces the analytic flat field
P3 = TET[[0, 1, 2]]
flat_nodes = np.array([P3[0], P3[1], P3[2], 0.5*(P3[0]+P3[1]), 0.5*(P3[1]+P3[2]), 0.5*(P3[2]+P3[0])])
flatmap = make_t6_surface_map(flat_nodes)
sigl = lambda p: 0.5 + 0.6 * p[0] - 0.4 * p[1]
print("curved_triangle_charge_field flat-limit vs the analytic flat field:")
for r in [np.array([0.3, 0.3, 0.7]), np.array([1.5, 0.5, 0.4])]:
    a = curved_triangle_charge_field(flatmap, r, sigl, nq=20)
    b = linear_triangle_charge_field(P3, r, 0.5, np.array([0.6, -0.4, 0.0]))
    print(f"  r={r.tolist()}  rel = {rel(a, b):.2e}")

# curved patch: singularity subtraction converges to a Duffy-refined reference
c = 0.15
nodes = np.array([[0,0,0],[1,0,0],[0,1,0],[0.5,0,c],[0.5,0.5,c],[0,0.5,c]], float)
smap = make_t6_surface_map(nodes)
sig = lambda p: 0.5 + 0.6 * p[0] - 0.4 * p[1] + 0.3 * p[2]


def curved_ref(surf_map, r, sigma_fn, nq=80):
    r = np.asarray(r, float); u0, v0 = _project_to_surface(surf_map, r)
    P0 = np.array([u0, v0]); corners = [np.array([0., 0.]), np.array([1., 0.]), np.array([0., 1.])]
    xs, ws = np.polynomial.legendre.leggauss(nq); xs = 0.5 * (xs + 1); ws = 0.5 * ws
    F = np.zeros(3)
    for a in range(3):
        B = corners[a]; C = corners[(a+1) % 3]
        area2 = abs((B[0]-P0[0])*(C[1]-P0[1]) - (B[1]-P0[1])*(C[0]-P0[0]))
        for i in range(nq):
            for j in range(nq):
                ss, tt = xs[i], xs[j]
                u, v = P0 + ss*((1-tt)*(B-P0) + tt*(C-P0)); jac = ss*area2
                x, xu, xv = surf_map(u, v); J = np.linalg.norm(np.cross(xu, xv)); dd = r - x
                F += ws[i]*ws[j]*jac*sigma_fn(x)*dd/np.linalg.norm(dd)**3*J
    return F


print("curved patch (bow c=0.15): singularity subtraction (nq=20) vs Duffy reference (nq=80):")
for h, tag in [(0.5, "far"), (0.205, "very near (surface bows to ~0.198)")]:
    r = np.array([0.3, 0.3, h])
    print(f"  h={h} ({tag})  rel = {rel(curved_triangle_charge_field(smap, r, sig, nq=20), curved_ref(smap, r, sig, nq=80)):.2e}")

curved_triangle_charge_field flat-limit vs the analytic flat field:
  r=[0.3, 0.3, 0.7]  rel = 7.64e-16
  r=[1.5, 0.5, 0.4]  rel = 1.40e-15
curved patch (bow c=0.15): singularity subtraction (nq=20) vs Duffy reference (nq=80):


  h=0.5 (far)  rel = 5.17e-11


  h=0.205 (very near (surface bows to ~0.198))  rel = 2.40e-05


## Curved-element VOLUME-charge field

> The `.md` listed the curved/distorted-hex VOLUME as the harder remaining piece.
> It has since landed as `curved_tet_volume_field` (exported from `radia.vim`,
> golden-locked), so this notebook demonstrates it.

A curved tet (`mesh.Curve(p)`, T10 quadratic map) is **not** its flat 4-corner
approximation: the edge bowing changes the volume-charge field by O(curvature).
The volume self-field is only *mildly* singular (the $s^2$ volume Jacobian
cancels $(r-x)/R^3$), so the robust route is **subdivision**: $1\!\to\!8$ (Bey
"red") refine the reference tet, map each reference sub-tet's 4 corners through
the curved map to a **flat** physical sub-tet, and sum the EXACT closed-form
`tet_volume_field_linear` over the sub-tets. `depth = 0` is the flat 4-corner tet
(ignores the bowing); increasing depth -> the true curved-tet field. Below: the
zero-bow **flat limit** (reproduces the flat closed form), and a strongly bowed
tet where depth-3 converges to a brute-force curved reference while depth-0 (flat)
misses badly.

In [13]:
corners = np.array([[0, 0, 0], [1.0, 0, 0], [0, 1.0, 0], [0, 0, 1.0]], float)
edge_pairs = [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]
mids = np.array([0.5 * (corners[i] + corners[j]) for (i, j) in edge_pairs])

# flat limit: zero edge bowing reproduces the analytic flat closed form
tmap_flat = make_t10_tet_map(np.vstack([corners, mids]))
print("curved_tet_volume_field flat-limit vs tet_volume_field_linear:")
for r in [np.array([2.0, 0.3, 0.2]), np.array([0.3, 0.3, 0.25])]:
    flat = tet_volume_field_linear(corners, r, 1.0, np.zeros(3))
    cv = curved_tet_volume_field(tmap_flat, r, lambda x: 1.0, depth=2)
    print(f"  r={r.tolist()}  rel = {rel(cv, flat):.2e}")

# strongly bowed tet: depth-3 converges to a brute-force curved reference
mids_bow = mids + 0.18 * (mids - corners.mean(0))
tmap = make_t10_tet_map(np.vstack([corners, mids_bow]))


def curved_tet_ref(tet_map, r, rho_fn, ng=14):
    xs, ws = np.polynomial.legendre.leggauss(ng); xs = 0.5 * (xs + 1); ws = 0.5 * ws
    H = np.zeros(3); hh = 1e-6
    for a, wa in zip(xs, ws):
        for b, wb in zip(xs, ws):
            for cc, wc in zip(xs, ws):
                xi, eta, zeta = a, b*(1-a), cc*(1-a)*(1-b); duffy = (1-a)**2*(1-b)
                x = tet_map(xi, eta, zeta)
                jx = (tet_map(xi+hh, eta, zeta) - tet_map(xi-hh, eta, zeta)) / (2*hh)
                jy = (tet_map(xi, eta+hh, zeta) - tet_map(xi, eta-hh, zeta)) / (2*hh)
                jz = (tet_map(xi, eta, zeta+hh) - tet_map(xi, eta, zeta-hh)) / (2*hh)
                detJ = abs(np.linalg.det(np.array([jx, jy, jz]).T)); dd = r - x
                H += wa*wb*wc*duffy*detJ*rho_fn(x)*dd/np.linalg.norm(dd)**3
    return H


r = np.array([2.0, 0.3, 0.2]); ref = curved_tet_ref(tmap, r, lambda x: 1.0, ng=14)
e0 = rel(curved_tet_volume_field(tmap, r, lambda x: 1.0, 0), ref)
e3 = rel(curved_tet_volume_field(tmap, r, lambda x: 1.0, 3), ref)
print(f"strongly-bowed tet at r={r.tolist()}:  depth-0 (flat) rel = {e0:.2e}   depth-3 rel = {e3:.2e}")

curved_tet_volume_field flat-limit vs tet_volume_field_linear:
  r=[2.0, 0.3, 0.2]  rel = 2.11e-13


  r=[0.3, 0.3, 0.25]  rel = 2.78e-15


strongly-bowed tet at r=[2.0, 0.3, 0.2]:  depth-0 (flat) rel = 3.83e-01   depth-3 rel = 6.50e-03


## Step 2 fast path -- the charge-coefficient assembly (`assemble_demag_field`)

Because the analytic degree-1/2 kernels are exact for **any** `r` (inside / on /
outside an element), the internal/near demag field of a solved HDiv-VIM
magnetization is a single uniform sum over all volume elements
(`rho = -div M`, linear for HDiv order <= 2 -> `_hdiv_tet_volfield_linear`) and
boundary faces (`sigma = M.n`, quadratic -> `_hdiv_quad_tri_field`) -- NO
self/near/far split. The coefficients are extracted once per element/face, then
the batched C++ kernel (`_hdiv_demag_field_batch`, one `ngcore::ParallelFor` over
the observation points) runs.

Capstone: on the uniform-M sphere, the assembly gives `-M/3` at the center
(to the flat-mesh faceting) and matches the external Gauss reference to **machine
precision** at external points.

In [14]:
# reuse the uniform-M sphere (mesh, gf, Mval) from Step 1
with ng.TaskManager():
    Ha_center = assemble_demag_field(mesh, gf, np.array([[0.0, 0.0, 0.0]]))
    Ha_ext = assemble_demag_field(mesh, gf, obs)
    Href = reconstruct_field_polynomial(mesh, gf, obs, quad=6)   # external Gauss reference

print(f"assemble_demag_field center H_z = {Ha_center[0,2]:.5e}   -M/3 = {-Mval/3:.5e}   "
      f"rel = {abs(Ha_center[0,2]+Mval/3)/(Mval/3):.2e}")
print("assemble_demag_field vs external Gauss reference (machine precision):")
for i, r in enumerate(obs):
    print(f"  obs r={r.tolist()}  rel = {rel(Ha_ext[i], Href[i]):.2e}")

assemble_demag_field center H_z = -1.66461e+05   -M/3 = -1.66667e+05   rel = 1.23e-03
assemble_demag_field vs external Gauss reference (machine precision):
  obs r=[0.0, 0.0, 2.0]  rel = 2.15e-14
  obs r=[0.0, 0.0, 3.0]  rel = 1.15e-14
  obs r=[2.0, 0.0, 0.0]  rel = 2.22e-14


## Notes / pitfalls (verify-first record)

- Use NGSolve's own geometry, not a hand-rolled affine map:
  `trafo = mesh.GetTrafo(ElementId(...))`,
  `for ip in IntegrationRule(mesh[ei].type, order)`, `mip = trafo(ip)`. Physical
  point = `mip.point`, physical quadrature weight = `ip.weight * mip.measure`
  (the MIP exposes `.point`, `.measure`, `.jacobi` -- **not** `GetMeasure()` /
  `GetJacobiDet()` / `.weight`). This makes the kernel handle tet **and** hex
  (and curved) for free.
- Surface charge `M.n`: build it as a CF
  `InnerProduct(gfM.Trace(), specialcf.normal(dim))` and evaluate at the boundary
  MIP -- `specialcf.normal` is the correct outward (and curved) normal; do not
  hand-roll the face normal from vertices (wrong sign / no curving).
- `div(gfM)(mip)` may return a 1-tuple -- extract the scalar.
- The field integrand is `1/r^2`-singular, so Step 1 is **external only**; an
  internal/near point needs the Step-2 singular-aware kernel. Do not use Step 1
  inside the body.

### Where this lives

- **Python reference:** [`src/radia/vim/_field.py`](../../src/radia/vim/_field.py)
  (all kernels above, exported from `radia.vim`).
- **C++ fast path:** `src/core/rad_hdiv_vim.cpp` (the degree-1/2 kernels +
  `_hdiv_demag_field_batch`), probed in `radia_pybind.cpp`.
- **Charges:** the same `rho = -div M` / `sigma = M.n` the HDiv-VIM forms in
  `ChargeGram` (the `B` map); this kernel is the field-at-a-point companion
  of the charge **Gram** `G`.
- **Golden tests:**
  [`validation_test/feec/test_hdiv_vim_poly_field.py`](../../validation_test/feec/test_hdiv_vim_poly_field.py)
  (36 locks). This notebook recomputes the representative ones live.